# 🪖 Train Helmet Detection Model

> **Đề tài:** Hệ thống Giám sát Phương tiện Giao thông – DATN_VTHUW  
> **Model:** YOLOv8s – Phát hiện có/không đội mũ bảo hiểm  
> **Dataset:** Helmet Detection datasets từ Roboflow Universe  
> **Output:** `helmet_detection.pt` → lưu Google Drive

⚡ **Bật GPU:** Runtime → Change runtime type → **T4 GPU**

---
### Các dataset helmet phù hợp:
| Dataset | Ảnh | Classes | Đặc điểm |
|---------|-----|---------|----------|
| Safety Helmet Detection (Roboflow) | 5,000+ | helmet, no-helmet | Xây dựng, công trường |
| Helmet Detection Motorcyclist | 3,000+ | with_helmet, without_helmet | **Phù hợp nhất cho DATN** |
| Hard Hat Workers (Kaggle) | 7,000+ | helmet, no_helmet, head | Đa dạng |

## 📦 Bước 1: Cài đặt & GPU

In [ ]:
!nvidia-smi
!pip install ultralytics roboflow --quiet

import ultralytics, torch
ultralytics.checks()
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## ☁️ Bước 2: Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/DATN_TrafficAI'
MODEL_DIR = f'{DRIVE_DIR}/models'
LOG_DIR   = f'{DRIVE_DIR}/logs'

for d in [MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'✅ Drive mounted. Models: {MODEL_DIR}')

## 📂 Bước 3: Tải Dataset Helmet

> Dataset tốt nhất cho bài toán phát hiện mũ bảo hiểm người đi xe máy ở Việt Nam

In [ ]:
# ==============================================================
# 🅰️ CÁCH A: Roboflow – Helmet Detection (xe máy, phù hợp nhất)
# ==============================================================

RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # <-- THAY VÀO ĐÂY

# Dataset options:
DATASETS = {
    # Option 1: Helmet/No-Helmet (motorcyclist focused)
    'motorcyclist': {
        'workspace': 'tesis-zjvfz',
        'project': 'helmet-no-helmet',
        'version': 2,
    },
    # Option 2: Safety Helmet Detection (broader, 5k+ images)
    'safety_helmet': {
        'workspace': 'roboflow-100',
        'project': 'safety-helmet-detection-ggnnr',
        'version': 2,
    },
    # Option 3: Hard Hat Detection (Kaggle-style via Roboflow)
    'hard_hat': {
        'workspace': 'joseph-nelson',
        'project': 'hard-hat-universe',
        'version': 14,
    },
}

# Chọn dataset (đổi key theo options trên)
CHOSEN = 'safety_helmet'  # <-- Thay: 'motorcyclist', 'safety_helmet', 'hard_hat'

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)

cfg = DATASETS[CHOSEN]
project = rf.workspace(cfg['workspace']).project(cfg['project'])
dataset = project.version(cfg['version']).download('yolov8')

DATASET_PATH = dataset.location
print(f'\n✅ Dataset [{CHOSEN}] tải về: {DATASET_PATH}')

In [ ]:
# ==============================================================
# 🅱️ CÁCH B: Kaggle – Hard Hat Workers Dataset (7,000+ ảnh)
# Cần: Kaggle account + API token
# ==============================================================

# from google.colab import files
# print('Upload file kaggle.json')
# files.upload()  # Upload kaggle.json từ https://www.kaggle.com/settings

# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install kaggle -q
# !kaggle datasets download -d andrewmvd/helmet-detection -p /content/datasets/helmet/
# !unzip -q /content/datasets/helmet/helmet-detection.zip -d /content/datasets/helmet/

# DATASET_PATH = '/content/datasets/helmet'
# print('Dataset structure:')
# !ls /content/datasets/helmet/

In [ ]:
# ==============================================================
# 🅲 CÁCH C: Tự gán nhãn với ảnh giao thông Việt Nam
# Dùng Roboflow Annotate (miễn phí) hoặc LabelImg
# ==============================================================

# Hướng dẫn:
# 1. Thu thập ảnh giao thông Việt Nam (webcam, internet)
# 2. Upload lên https://app.roboflow.com → Create Dataset
# 3. Annotate online với 2 classes: 'helmet', 'no_helmet'
# 4. Export theo YOLOv8 format
# 5. Tải về và upload lên Drive

# DATASET_PATH = '/content/drive/MyDrive/DATN_TrafficAI/datasets/helmet'

print('Hướng dẫn gán nhãn:')
print('1. Dùng Roboflow Annotate: https://app.roboflow.com')
print('2. Hoặc LabelImg: pip install labelImg && labelImg')
print('3. Classes: helmet (0), no_helmet (1)')

In [ ]:
# Tìm file data.yaml và xem nội dung
import glob

yaml_files = glob.glob(f'{DATASET_PATH}/**/*.yaml', recursive=True)
YAML_FILE = yaml_files[0] if yaml_files else None

if YAML_FILE:
    print(f'📄 Config: {YAML_FILE}\n')
    with open(YAML_FILE) as f:
        content = f.read()
    print(content)

    # Đếm ảnh
    n_train = len(glob.glob(f'{DATASET_PATH}/train/images/*'))
    n_val   = len(glob.glob(f'{DATASET_PATH}/valid/images/*'))
    print(f'Train: {n_train} | Val: {n_val}')
else:
    print('⚠️  Chưa tìm thấy data.yaml – hãy tải dataset ở bước trên')

In [ ]:
# Hiển thị vài ảnh mẫu từ dataset
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

sample_images = glob.glob(f'{DATASET_PATH}/train/images/*')[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
plt.suptitle('Ảnh mẫu từ Dataset Helmet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔄 Bước 3b: Chuẩn hóa nhãn (nếu cần)

> Một số dataset dùng tên class khác nhau (helmet/no_helmet, with_helmet/without_helmet, head/hardhat)
> Script này chuẩn hóa về: class 0 = helmet, class 1 = no_helmet

In [ ]:
import yaml

# Đọc class names từ data.yaml
with open(YAML_FILE) as f:
    data_cfg = yaml.safe_load(f)

current_names = data_cfg.get('names', [])
print(f'Classes hiện tại: {current_names}')

# Mapping tên class → index chuẩn
HELMET_NAMES   = ['helmet', 'hardhat', 'hard_hat', 'with_helmet', 'has_helmet', 'Helmet']
NOHELMET_NAMES = ['no_helmet', 'no-helmet', 'without_helmet', 'nohelmet', 'head', 'no_hardhat']

# Tạo mapping: class_id_cũ → class_id_mới
remap = {}
for idx, name in enumerate(current_names):
    if name in HELMET_NAMES:
        remap[idx] = 0   # → helmet
    elif name in NOHELMET_NAMES:
        remap[idx] = 1   # → no_helmet

print(f'Mapping: {remap}')

# Nếu đã đúng format (0=helmet, 1=no_helmet) thì không cần remap
if remap == {0: 0, 1: 1} or not remap:
    print('✅ Nhãn đã đúng chuẩn, không cần remap')
else:
    print(f'⚠️  Cần remap nhãn, sẽ thực hiện ở cell tiếp theo')

In [ ]:
# Cập nhật data.yaml nếu cần
data_cfg['names'] = ['helmet', 'no_helmet']
data_cfg['nc'] = 2

# Fix đường dẫn trong yaml
data_cfg['path'] = DATASET_PATH
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'

# Lưu lại
with open(YAML_FILE, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, allow_unicode=True)

print('✅ data.yaml đã cập nhật:')
with open(YAML_FILE) as f:
    print(f.read())

## 🚀 Bước 4: Huấn luyện Model

In [ ]:
from ultralytics import YOLO

# Helmet cần model nhỏ hơn vì chỉ detect vùng đầu nhỏ
# YOLOv8s là lựa chọn tốt
model = YOLO('yolov8s.pt')

results = model.train(
    data=YAML_FILE,
    epochs=80,            # Nhiều hơn vì dataset nhỏ hơn vehicle
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    name='helmet_detection',
    project='/content/runs',
    save=True,
    save_period=10,
    exist_ok=True,
    amp=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    # Augmentation đặc biệt cho helmet detection
    fliplr=0.5,
    flipud=0.0,           # Không lật ngược (người không đứng ngược)
    mosaic=1.0,
    mixup=0.0,            # Không mixup vì class nhỏ (đầu người)
    degrees=10.0,         # Xoay ±10 độ
    translate=0.2,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

print('\n✅ Training hoàn tất!')

## 📊 Bước 5: Đánh giá Model

In [ ]:
from IPython.display import Image as IPImage, display
import os

run_dir = '/content/runs/helmet_detection'

for img_name in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    path = f'{run_dir}/{img_name}'
    if os.path.exists(path):
        print(f'\n--- {img_name} ---')
        display(IPImage(path, width=800))

In [ ]:
best_model = YOLO(f'{run_dir}/weights/best.pt')
metrics = best_model.val()

print('\n=== KẾT QUẢ HELMET DETECTION ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

# Per-class metrics
print('\nPer-class AP:')
for name, ap in zip(['helmet', 'no_helmet'], metrics.box.maps):
    print(f'  {name}: {ap:.4f}')

In [ ]:
# Test inference trên ảnh val
import glob, random
from IPython.display import Image as IPImage, display

val_imgs = glob.glob(f'{DATASET_PATH}/valid/images/*')[:5]

for img_path in val_imgs:
    result = best_model.predict(
        img_path,
        conf=0.45,
        save=True,
        project='/content/predictions/helmet',
        exist_ok=True,
    )

pred_imgs = glob.glob('/content/predictions/helmet/**/*.jpg', recursive=True)
for img in pred_imgs[:3]:
    display(IPImage(img, width=640))

## 💾 Bước 6: Lưu Model

In [ ]:
import shutil

best_pt = f'/content/runs/helmet_detection/weights/best.pt'
dest    = f'{MODEL_DIR}/helmet_detection.pt'

shutil.copy2(best_pt, dest)
print(f'✅ Model: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)')

# Lưu logs
shutil.copytree('/content/runs/helmet_detection', f'{LOG_DIR}/helmet_detection', dirs_exist_ok=True)

print('\n🎉 Hoàn tất!')
print('Tải về: Google Drive → DATN_TrafficAI/models/helmet_detection.pt')
print('Copy vào: DATN_VTHUW/models/helmet_detection.pt')

In [ ]:
# Download thẳng về máy
from google.colab import files
# files.download(f'/content/runs/helmet_detection/weights/best.pt')